# === CELL 1 (Markdown Guide) ===
## VHS24K: Wedding VHS/DVD to 4K Restoration on Kaggle (2x NVIDIA T4)

- **CELL 2**: Check Dual T4 GPUs & FFmpeg
- **CELL 3**: Install `VHS24K`, `Real-ESRGAN`, `BasicSR`, `GFPGAN`, and AI model weights
- **CELL 4**: Download `Wedding` `.VOB` files from Google Drive (pre-configured for link-shared `Wedding` folder)
- **CELL 5**: Render a **15-second 4K Test Clip** (`--test-clip 15`) to `/kaggle/working/output`
- **CELL 6**: Run the **Full Resumable Dual-T4 Pipeline** on all Wedding footage

In [ ]:
# === CELL 2: Verify Dual T4 GPUs & FFmpeg ===
!nvidia-smi
!ffmpeg -version | head -n 3

In [ ]:
# === CELL 3: Install VHS24K, Real-ESRGAN, BasicSR, GFPGAN & Model Weights ===
import importlib
import os
import sys
from pathlib import Path

WORK_ROOT = Path("/kaggle/working")
VHS24K_DIR = WORK_ROOT / "VHS24K"
REALESRGAN_DIR = WORK_ROOT / "Real-ESRGAN"

# 1. Clone or pull latest VHS24K and add to Python path
if VHS24K_DIR.exists():
    !git -C {VHS24K_DIR} pull
else:
    !git clone https://github.com/claytonjfaria-ai/VHS24K.git {VHS24K_DIR}
!pip install -q -e {VHS24K_DIR}
if str(VHS24K_DIR) not in sys.path:
    sys.path.insert(0, str(VHS24K_DIR))
os.environ["PYTHONPATH"] = f"{VHS24K_DIR}:{REALESRGAN_DIR}:{os.environ.get('PYTHONPATH', '')}"
importlib.invalidate_caches()

# 2. Install patched BasicSR, Real-ESRGAN, GFPGAN, and model weights
from vhs24k.kaggle_setup import setup_kaggle_environment
setup_kaggle_environment(REALESRGAN_DIR)

In [ ]:
# === CELL 4: Download Wedding Video Files from Google Drive ===
import sys
from pathlib import Path
if "/kaggle/working/VHS24K" not in sys.path:
    sys.path.insert(0, "/kaggle/working/VHS24K")

from vhs24k.drive_io import download_wedding_files

INPUT_DIR = Path("/kaggle/working/input")
WORK_DIR = Path("/kaggle/working/work")
OUTPUT_DIR = Path("/kaggle/working/output")
SOURCE_MODE = "vob"  # "vob" (recommended un-recompressed DVD) or "mp4"

downloaded = download_wedding_files(
    dest_dir=INPUT_DIR,
    source_type=SOURCE_MODE,
    method="gdown",  # Uses link-shared Wedding folder directly
)
print("Downloaded files:", [p.name for p in downloaded])

!PYTHONPATH=/kaggle/working/VHS24K python -m vhs24k.cli inspect --input-dir {INPUT_DIR} --source-mode {SOURCE_MODE}

In [ ]:
# === CELL 5: Render 15-Second 4K Test Clip (--test-clip 15) ===
!git -C /kaggle/working/VHS24K pull
!PYTHONPATH=/kaggle/working/VHS24K:/kaggle/working/Real-ESRGAN python -m vhs24k.cli run \
    --input-dir /kaggle/working/input \
    --work-dir /kaggle/working/work_test \
    --output-dir /kaggle/working/output \
    --source-mode vob \
    --model realesr-general-x4v3 \
    --4k-mode pillarbox_uhd \
    --num-gpus 2 \
    --test-clip 15 \
    --realesrgan-dir /kaggle/working/Real-ESRGAN

!ls -lh /kaggle/working/output

In [ ]:
# === CELL 6: Run Full Resumable Dual-T4 4K Pipeline ===
!PYTHONPATH=/kaggle/working/VHS24K:/kaggle/working/Real-ESRGAN python -m vhs24k.cli run \
    --input-dir /kaggle/working/input \
    --work-dir /kaggle/working/work \
    --output-dir /kaggle/working/output \
    --source-mode vob \
    --model realesr-general-x4v3 \
    --chunk-seconds 60 \
    --4k-mode pillarbox_uhd \
    --num-gpus 2 \
    --realesrgan-dir /kaggle/working/Real-ESRGAN

!ls -lh /kaggle/working/output